In [ ]:
from brainviz.data.loader import extract_data_slices
from brainviz.data.dataset import get_dataloader, BrainSliceDataset
from brainviz.visualize import show_slice, show_sample, plot_3d_segmentation_interactive, get_patient_volume
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

torch.cuda.is_available()

In [ ]:
x, y = extract_data_slices("../dataset/train/subject-1")

print(x.shape, y.shape)
print(x[0], y)

In [ ]:
dataset = get_dataloader("../dataset/train", batch_size=4, shuffle=True, min_foreground_ratio=0.17).dataset
x, y = dataset[0]
print(x.shape, y.shape)
show_sample(dataset)

In [ ]:
full_dataset = BrainSliceDataset("../dataset/train", axis=2, modality="T1")
y = get_patient_volume(full_dataset, "subject-1")
plot_3d_segmentation_interactive(y, classes=[2, 3])

In [ ]:
plot_3d_segmentation_interactive(y)                 # rotatif, toutes les classes
#plot_3d_segmentation_interactive(y, classes=[2, 3])  # sans le LCR, plus lisible

# Stats

In [ ]:
def plot_threshold_distribution(dataset, thresholds=None):
    """Trace le % de tranches conservées selon le seuil min_foreground_ratio.

    dataset : BrainSliceDataset non filtré (min_foreground_ratio=0.0), pour avoir
    la distribution complète de dataset.foreground_ratio sur laquelle balayer les seuils.
    thresholds : valeurs de seuil à tester, par défaut np.linspace(0, 0.3, 31).
    """
    if thresholds is None:
        thresholds = np.linspace(0, 0.3, 31)
    ratios = dataset.foreground_ratio.numpy()
    n_total = len(dataset)
    counts = [(ratios >= t).sum() for t in thresholds]
    df = pd.DataFrame({
        "threshold": thresholds,
        "n_slices": counts,
        "pct_slices": [100 * c / n_total for c in counts],
    })

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.lineplot(data=df, x="threshold", y="pct_slices", marker="o", ax=ax)
    ax.set_xlabel("min_foreground_ratio")
    ax.set_ylabel("% de tranches conservées")
    ax.set_title(f"Tranches conservées selon le seuil (sur {n_total} tranches)")
    fig.tight_layout()
    plt.close(fig)
    return fig


def plot_threshold_distribution_by_patient(dataset, thresholds=None):
    """Même courbe que plot_threshold_distribution, mais une ligne par patient.

    dataset : BrainSliceDataset non filtré, exposant subject_ids et foreground_ratio.
    thresholds : valeurs de seuil à tester, par défaut np.linspace(0, 0.3, 31).
    """
    if thresholds is None:
        thresholds = np.linspace(0, 0.3, 31)
    ratios = dataset.foreground_ratio.numpy()
    subjects = dataset.subject_ids

    rows = []
    for subject in sorted(np.unique(subjects), key=lambda s: int(s.split("-")[1])):
        subject_ratios = ratios[subjects == subject]
        n_total = len(subject_ratios)
        for t in thresholds:
            rows.append({
                "threshold": t,
                "subject": subject,
                "pct_slices": 100 * (subject_ratios >= t).sum() / n_total,
            })
    df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=df, x="threshold", y="pct_slices", hue="subject", marker="o", ax=ax)
    ax.set_xlabel("min_foreground_ratio")
    ax.set_ylabel("% de tranches conservées")
    ax.set_title("Tranches conservées selon le seuil, par patient")
    ax.legend(title="sujet", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    plt.close(fig)
    return fig

In [ ]:
plot_threshold_distribution(full_dataset)

In [ ]:
plot_threshold_distribution_by_patient(full_dataset)

# Crop du fond (bounding box du cerveau)

**Idée** : iSeg-2017 est déjà *skull-strippé* — le fond (hors cerveau) vaut exactement 0 sur T1 et T2, sur un champ de vue (FOV) bien plus grand que le cerveau lui-même (volume `(144, 192, 256)` par sujet). On envoie donc au modèle beaucoup de tranches/pixels qui ne contiennent aucune information utile.

**Vérification empirique (10 sujets d'entraînement)** : le masque d'intensité `(T1 > 0) | (T2 > 0)` est **identique voxel à voxel** au masque `label != 0`, sur les 10 sujets, sans une seule exception. Un simple seuillage d'intensité suffit donc à isoler exactement le cerveau, sans risque de rogner du vrai tissu — au moins sur ce dataset (déjà skull-strippé, cohorte homogène).

**Implémentation** (`brainviz.data.loader`, activable via `crop=True` / `--crop`) :
- bbox 3D par sujet, calculée sur l'union des voxels non nuls de T1 et T2 (`_brain_bbox`),
- marge de sécurité de `crop_margin` voxels (défaut 4), pour tolérer un sujet légèrement atypique — en particulier les sujets du split *test* (11-23), qui n'ont pas de label pour vérifier la bbox directement,
- **taille de canvas commune à toute la cohorte** (`compute_crop_size`, appelé dans `BrainSliceDataset` avant l'extraction) : chaque sujet a une bbox de taille différente, donc pour pouvoir empiler leurs tranches dans un seul tenseur il faut leur imposer la même taille — le max de la cohorte, arrondi au multiple de 8 supérieur (compatible `depth=3` du `CompactUNet`). Les sujets plus petits que ce max reçoivent un peu de padding résiduel, mais bien moins que sans crop.

**Limite à garder en tête** : la vérification voxel-à-voxel ci-dessus ne porte que sur les 10 sujets *train* (les seuls avec label). Elle suppose que les sujets *test* partagent le même pipeline de skull-stripping — raisonnable pour une cohorte unique de challenge, mais pas vérifiable directement.

*Note sur le tableau de comparaison ci-dessous : `canvas_crop` y est calculé par sujet indépendamment (bbox propre à chaque sujet), pour montrer la taille "naturelle" de chacun. En entraînement, `BrainSliceDataset` impose la taille commune la plus grande de la cohorte (160×160 sur `dataset/train`) à tous les sujets.*

In [ ]:
from pathlib import Path

import nibabel as nib

from brainviz.data.loader import _brain_bbox


def _load_volume(path):
    return np.asarray(nib.load(str(path)).dataobj).squeeze()


rows = []
for subject_dir in sorted(Path("../dataset/train").iterdir(), key=lambda p: int(p.name.split("-")[1])):
    t1 = _load_volume(subject_dir / "T1.img")
    t2 = _load_volume(subject_dir / "T2.img")
    label = _load_volume(subject_dir / "label.img")

    intensity_mask = (t1 > 0) | (t2 > 0)
    label_mask = label != 0

    lo, hi = _brain_bbox(t1, t2, margin=0)
    rows.append({
        "subject": subject_dir.name,
        "shape": t1.shape,
        "bbox_size": tuple((hi - lo + 1).tolist()),
        "masques_identiques": bool(np.array_equal(intensity_mask, label_mask)),
    })

pd.DataFrame(rows)

In [ ]:
import matplotlib.patches as mpatches

from brainviz.data.loader import _crop_to_bbox

subject_dir = Path("../dataset/train/subject-1")
t1 = _load_volume(subject_dir / "T1.img")
lo, hi = _brain_bbox(t1, _load_volume(subject_dir / "T2.img"), margin=4)

mid_axis2 = (lo[2] + hi[2]) // 2
slice_full = t1[:, :, mid_axis2]
cropped = _crop_to_bbox(t1, lo, hi)
slice_cropped = cropped[:, :, cropped.shape[2] // 2]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(slice_full, cmap="gray")
axes[0].add_patch(mpatches.Rectangle(
    (lo[1], lo[0]), hi[1] - lo[1], hi[0] - lo[0], edgecolor="red", facecolor="none", linewidth=2,
))
axes[0].set_title(f"T1 complet {tuple(slice_full.shape)} + bbox cerveau (marge=4)")
axes[0].axis("off")

axes[1].imshow(slice_cropped, cmap="gray")
axes[1].set_title(f"T1 cropé {tuple(slice_cropped.shape)}")
axes[1].axis("off")
fig.tight_layout()

In [ ]:
from brainviz.data.loader import extract_data_slices

rows = []
for subject_dir in sorted(Path("../dataset/train").iterdir(), key=lambda p: int(p.name.split("-")[1])):
    data, _ = extract_data_slices(subject_dir, axes=[2], scaling="padding", crop=False)
    data_c, _ = extract_data_slices(subject_dir, axes=[2], scaling="padding", crop=True)
    rows.append({
        "subject": subject_dir.name,
        "n_tranches_sans_crop": data.shape[0],
        "n_tranches_crop": data_c.shape[0],
        "canvas_sans_crop": tuple(data.shape[-2:]),
        "canvas_crop": tuple(data_c.shape[-2:]),
        "ratio_pixels": round(
            (data_c.shape[-1] * data_c.shape[-2]) / (data.shape[-1] * data.shape[-2]), 3
        ),
    })

df_crop = pd.DataFrame(rows)
df_crop

## Impact sur l'entraînement

Comparaison contrôlée entre le pipeline actuel (tranches paddées à 256×256) et l'entrée cropée : même seed (0), mêmes hyperparamètres (`base_channels=16`, `depth=3`, `lr=1e-3`, `batch_size=16`), 10 epochs, même split val (subject-9, subject-10). Résumés produits par `train.py` (`--crop` ou non), copiés dans `report/baseline_nocrop_summary.json` et `report/crop_summary.json`.

In [ ]:
import json
import statistics


def load_summary(name):
    with open(f"../report/{name}") as f:
        return json.load(f)


def best_epoch_row(summary, label):
    best = max(summary["history"], key=lambda h: h["mean_dice_fg"])
    # médiane plutôt que moyenne : robuste aux epochs polluées par de la contention CPU
    # (ex. un autre process tournant en parallèle sur la même machine).
    median_epoch_time = statistics.median(h["elapsed_s"] for h in summary["history"])
    return {
        "run": label,
        "canvas": tuple(summary["image_size"]),
        "epoch (meilleur)": best["epoch"],
        "Dice moyen (fg)": round(best["mean_dice_fg"], 3),
        "temps/epoch médian (s)": round(median_epoch_time, 1),
    }


baseline_summary = load_summary("baseline_nocrop_summary.json")
crop_summary = load_summary("crop_summary.json")

df_training = pd.DataFrame([
    best_epoch_row(baseline_summary, "sans crop"),
    best_epoch_row(crop_summary, "avec crop"),
])
df_training

In [ ]:
from IPython.display import Image, display

display(Image("../report/figures/baseline_nocrop_10ep_learning_curves.png"))
display(Image("../report/figures/crop_10ep_learning_curves.png"))

### Mise en garde sur `train_loss` / `val_loss`

Ces deux valeurs **ne sont pas comparables** entre les deux runs. `nn.CrossEntropyLoss()` moyenne la perte **par pixel**, pas par classe : le fond est la classe la plus facile (Dice ≈ 1,0000 dès l'epoch 5 dans les deux runs), donc sa contribution à la perte est quasi nulle une fois apprise — plus il y a de pixels de fond, plus la moyenne globale est tirée vers le bas, indépendamment de la difficulté réelle des autres classes.

C'est le même piège méthodologique que le biais du Dice moyenné par batch (voir `report/rapport_entrainement.md`) : une métrique moyennée par pixel/par batch peut être dominée par la classe ou le cas majoritaire plutôt que refléter la vraie performance. Le Dice foreground (moyenné par classe, pas par pixel) reste la métrique valide pour comparer crop vs sans crop.

In [ ]:
nocrop_bg_fraction = 1 - BrainSliceDataset("../dataset/train", axis=2, modality="T1T2", crop=False).foreground_ratio.mean().item()
crop_bg_fraction = 1 - BrainSliceDataset("../dataset/train", axis=2, modality="T1T2", crop=True).foreground_ratio.mean().item()

best_baseline = max(baseline_summary["history"], key=lambda h: h["mean_dice_fg"])
best_crop = max(crop_summary["history"], key=lambda h: h["mean_dice_fg"])

pd.DataFrame([
    {
        "run": "sans crop",
        "fraction pixels de fond": round(nocrop_bg_fraction, 3),
        "train_loss (meilleure epoch)": best_baseline["train_loss"],
        "val_loss (meilleure epoch)": best_baseline["val_loss"],
        "Dice moyen (fg)": round(best_baseline["mean_dice_fg"], 3),
    },
    {
        "run": "avec crop",
        "fraction pixels de fond": round(crop_bg_fraction, 3),
        "train_loss (meilleure epoch)": best_crop["train_loss"],
        "val_loss (meilleure epoch)": best_crop["val_loss"],
        "Dice moyen (fg)": round(best_crop["mean_dice_fg"], 3),
    },
])

## Overhead du crop à l'inférence

Le gain observé à l'entraînement (~7,6× par epoch) s'explique en bonne partie par l'amortissement du coût de calcul de la bbox — fait une seule fois, à la construction du dataset — sur de nombreux batches et epochs. À l'inférence, un seul passage forward par sujet, la question se pose différemment : **l'overhead de calcul de la bbox est-il compensé par le gain d'un forward pass plus petit ?**

Mesure directe sur `subject-1` (CPU, `CompactUNet` en mode `eval()`, `torch.no_grad()`), en isolant :
- le coût **partagé** (chargement T1/T2 depuis le disque, identique avec ou sans crop),
- l'**overhead spécifique au crop** (seuillage + bbox `_brain_bbox`, découpe `_crop_to_bbox`),
- le coût du **forward pass**, avec et sans crop.

In [ ]:
import time

from brainviz.data.loader import _brain_bbox, _crop_to_bbox, _load_volume, _round_up, _scale_slices
from brainviz.models import CompactUNet

bench_subject_dir = Path("../dataset/train/subject-1")
N_REPEAT = 5


def timeit(fn, n=N_REPEAT):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        out = fn()
        times.append(time.perf_counter() - t0)
    return out, times


def med(xs):
    return sorted(xs)[len(xs) // 2]


# coût partagé : chargement T1/T2 (identique avec ou sans crop)
_, t_load = timeit(lambda: (_load_volume(bench_subject_dir / "T1.img"), _load_volume(bench_subject_dir / "T2.img")))
t1_vol = _load_volume(bench_subject_dir / "T1.img")
t2_vol = _load_volume(bench_subject_dir / "T2.img")

# overhead crop : seuillage + bbox
(bbox_lo, bbox_hi), t_bbox = timeit(lambda: _brain_bbox(t1_vol, t2_vol, margin=4))

# overhead crop : découpe (vue numpy, quasi gratuite)
_, t_crop_op = timeit(lambda: (_crop_to_bbox(t1_vol, bbox_lo, bbox_hi), _crop_to_bbox(t2_vol, bbox_lo, bbox_hi)))
t1_crop, t2_crop = _crop_to_bbox(t1_vol, bbox_lo, bbox_hi), _crop_to_bbox(t2_vol, bbox_lo, bbox_hi)

bench_size_full = max(t1_vol.shape)
bench_size_crop = _round_up(max(t1_crop.shape), 8)


def make_t1t2_batch(t1_arr, t2_arr, target_size):
    t1_axis = np.moveaxis(t1_arr, 2, 0)
    t2_axis = np.moveaxis(t2_arr, 2, 0)
    data_axis = np.stack([t1_axis, t2_axis], axis=1)
    data = _scale_slices(torch.from_numpy(data_axis).float(), target_size, "padding", "bilinear")
    t1c, t2c = data[:, 0:1], data[:, 1:2]
    ratio = t1c / (t2c + 1e-6)
    ratio = ratio / ratio.abs().amax(dim=(1, 2, 3), keepdim=True).clamp(min=1e-6)
    return torch.cat([t1c, t2c, ratio], dim=1)


batch_full = make_t1t2_batch(t1_vol, t2_vol, bench_size_full)
batch_crop = make_t1t2_batch(t1_crop, t2_crop, bench_size_crop)

torch.manual_seed(0)
bench_model = CompactUNet(in_channels=3, num_classes=4, base_channels=16, depth=3)
bench_model.eval()


def bench_forward(batch, batch_size=16, n=N_REPEAT):
    times = []
    with torch.no_grad():
        for _ in range(n):
            t0 = time.perf_counter()
            for i in range(0, batch.shape[0], batch_size):
                bench_model(batch[i:i + batch_size])
            times.append(time.perf_counter() - t0)
    return times


t_fwd_full = bench_forward(batch_full)
t_fwd_crop = bench_forward(batch_crop)

pd.DataFrame([
    {"étape": "chargement T1+T2 (partagé)", "temps (ms)": round(med(t_load) * 1000, 1)},
    {"étape": "overhead crop — bbox (seuillage)", "temps (ms)": round(med(t_bbox) * 1000, 1)},
    {"étape": "overhead crop — découpe", "temps (ms)": round(med(t_crop_op) * 1000, 2)},
    {
        "étape": f"forward pass sans crop ({bench_size_full}×{bench_size_full}, {batch_full.shape[0]} tranches)",
        "temps (ms)": round(med(t_fwd_full) * 1000, 1),
    },
    {
        "étape": f"forward pass avec crop ({bench_size_crop}×{bench_size_crop}, {batch_crop.shape[0]} tranches)",
        "temps (ms)": round(med(t_fwd_crop) * 1000, 1),
    },
])

In [ ]:
overhead_crop = med(t_bbox) + med(t_crop_op)
gain_forward = med(t_fwd_full) - med(t_fwd_crop)
total_nocrop = med(t_fwd_full)
total_crop = overhead_crop + med(t_fwd_crop)

print(f"overhead crop total (bbox + découpe) : {overhead_crop * 1000:.1f} ms")
print(f"gain forward pass                    : {gain_forward * 1000:.1f} ms")
print(f"overhead / gain                      : {100 * overhead_crop / gain_forward:.2f} %")
print(f"speedup net inférence (1 sujet, hors chargement partagé) : {total_nocrop / total_crop:.2f}x")

**Conclusion** : l'overhead du crop (seuillage + bbox, quelques dizaines de ms) est négligeable devant le gain du forward pass (plusieurs secondes) — moins de 1 % du gain, systématiquement. Le speedup net à l'inférence est du même ordre de grandeur que celui observé par epoch à l'entraînement (~7-8×, la valeur exacte varie légèrement d'une exécution à l'autre selon le bruit machine), malgré l'absence d'amortissement sur plusieurs epochs : le déséquilibre de coût entre un seuillage numpy vectorisé sur des voxels (O(n), rapide) et un réseau de convolutions sur des pixels est structurel, pas un artefact du nombre d'epochs.

**Reste à faire pour une vraie inférence sur le split test** : replacer la prédiction dans le repère du volume d'origine (pad inverse de la bbox vers la shape d'origine) pour l'exporter ou l'évaluer — une opération du même ordre de grandeur négligeable que la découpe elle-même.